## 🚀 Carlos Hinrichsen | Mock GPT-4o-mini Call Analysis

### 📋 Executive Summary
Problem: Analyze 100+ call transcripts → structured insights (issue/sentiment/action) for agent routing/dashboards.

Solution: Classical NLP + OpenAI GPT-4o-mini (mocked for demo). 15% NaN handling.

Business Value: Outages → tech queue (21% calls). $0.75/1000 calls.

#### 1. API Key + Imports

In [ ]:
# === SETUP: API KEY + LIBRARIES ===
import os
import openai
import pandas as pd
import numpy as np
import json
import warnings
import kaleido
warnings.filterwarnings('ignore')

# YOUR OPENAI KEY HERE (replace 'sk-...')
API_KEY = ""  # Paste from platform.openai.com/api-keys
openai.api_key = API_KEY

print("✅ OpenAI Setup Complete")
print(f"Key loaded: {'✅' if API_KEY != 'sk-your-actual-key-here' else '❌ Add your key!'}")

✅ OpenAI Setup Complete
Key loaded: ✅


#### 2. Load Data

In [22]:
df = pd.read_csv('synthetic_charter_calls_100.csv')
print(f"✅ Loaded {len(df)} calls")

✅ Loaded 100 calls


In [23]:
print(f"Shape: {df.shape}")
print(f"NaN transcripts: {df['transcript'].isna().mean():.1%}")
print("\nSample:")
print(df.head())

Shape: (100, 4)
NaN transcripts: 15.0%

Sample:
   call_id                                         transcript  duration  \
0        1    payment issue. Customer very upset. Tech visit?     271.0   
1        2  Spanish: quiero mejorar el plan. Cliente satis...     524.0   
2        3  very upset no internet. Customer very upset. T...     297.0   
3        4  payment issue. Customer very upset. Wants credit.     188.0   
4        5  tech support needed. Customer satisfied. After...     144.0   

  language  
0       en  
1       es  
2       es  
3       en  
4       en  


#### 3. Clean Data

In [24]:
df_clean = df.copy()
df_clean['transcript'] = df_clean['transcript'].fillna('Silent call')  # Business fill

In [26]:
# Preview outage keywords
df_clean['has_outage'] = df_clean['transcript'].str.contains('outage|no funciona|apagon', case=False, na=False)
print("🧹 Post-Clean Preview:")
print(df_clean[['call_id', 'transcript', 'has_outage', 'language']].head(20))
print(f"\n🚨 Outages detected: {df_clean['has_outage'].sum()} ({df_clean['has_outage'].mean():.1%})")

🧹 Post-Clean Preview:
    call_id                                         transcript  has_outage  \
0         1    payment issue. Customer very upset. Tech visit?       False   
1         2  Spanish: quiero mejorar el plan. Cliente satis...       False   
2         3  very upset no internet. Customer very upset. T...       False   
3         4  payment issue. Customer very upset. Wants credit.       False   
4         5  tech support needed. Customer satisfied. After...       False   
5         6  normal inquiry. Customer very upset. Repeated ...       False   
6         7  cancel service please. Customer satisfied. Tec...       False   
7         8                                        Silent call       False   
8         9   normal inquiry. Customer very upset. Tech visit?       False   
9        10  Spanish: soporte tecnico. Cliente muy molesto....       False   
10       11  satisfied with service. Customer very upset. R...       False   
11       12  Spanish: internet no funciona

#### 4. OpenAI Function

In [29]:
# === LIVE OPENAI API ===
def analyze_call_live(transcript: str) -> dict:
    """Real GPT-4o-mini API call."""
    try:
        response = openai.chat.completions.create(
            model="gpt-4o-mini",  # Fast/cheap
            messages=[
                {"role": "system", "content": """Charter call analyst. Output ONLY JSON:
                {"issue": "billing|outage|upgrade|other", "sentiment": "positive|negative|neutral", 
                 "action": "credit|tech|cancel|upsell|none", "summary": "1 sentence exactly from transcript."}"""},
                {"role": "user", "content": transcript}
            ],
            temperature=0.1,  # Consistent
            max_tokens=80
        )
        content = response.choices[0].message.content.strip()
        return json.loads(content) if content.startswith('{') else {"error": "non-json"}
    except Exception as e:
        print(f"API Error: {e}")
        return {"issue": "api_error", "sentiment": "neutral", "action": "none", "summary": str(e)[:100]}

#### 5. Run Analysis

In [44]:
print("🔴 LIVE GPT-4o-mini ($)")
df['analysis'] = df_clean['transcript'].apply(analyze_call_live)  # Smaller for cost



🔴 LIVE GPT-4o-mini ($)


In [45]:
print("✅ Analysis complete!")
print(df[['call_id', 'transcript', 'analysis']].head(20))

✅ Analysis complete!
    call_id                                         transcript  \
0         1    payment issue. Customer very upset. Tech visit?   
1         2  Spanish: quiero mejorar el plan. Cliente satis...   
2         3  very upset no internet. Customer very upset. T...   
3         4  payment issue. Customer very upset. Wants credit.   
4         5  tech support needed. Customer satisfied. After...   
5         6  normal inquiry. Customer very upset. Repeated ...   
6         7  cancel service please. Customer satisfied. Tec...   
7         8                                                NaN   
8         9   normal inquiry. Customer very upset. Tech visit?   
9        10  Spanish: soporte tecnico. Cliente muy molesto....   
10       11  satisfied with service. Customer very upset. R...   
11       12  Spanish: internet no funciona desde ayer. Clie...   
12       13  Spanish: cancelar servicio por favor. Cliente ...   
13       14  normal inquiry. Customer satisfied. Wants 

#### 6. Results Table

In [46]:
# === 5. RESULTS TABLE ===
analysis_df = pd.json_normalize(df['analysis'])
analysis_df['call_id'] = df['call_id']
analysis_df['duration'] = df['duration']

# Clean columns
cols = ['call_id', 'duration', 'issue', 'sentiment', 'action', 'summary']
result_df = analysis_df[cols].head(30)

print("🤖 GPT Analysis Results:")
print(result_df.to_string(index=False))

🤖 GPT Analysis Results:
 call_id  duration   issue sentiment action                                                                                                     summary
       1     271.0 billing  negative   tech                           Customer is very upset about a payment issue and is inquiring about a tech visit.
       2     524.0 upgrade  positive upsell                                                                                         Cliente satisfecho.
       3     297.0  outage  negative   tech                                                          Customer is very upset due to no internet service.
       4     188.0 billing  negative credit                                            Customer is very upset about a payment issue and wants a credit.
       5     144.0   other  positive   tech                                        Customer satisfied after receiving tech support following the storm.
       6     287.0   other  negative   none                     

#### 7. Sentient Graphics

In [64]:

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
import json

# Ensure analysis column exists (from your GPT)
if 'analysis' not in df.columns:
    print("⚠️ Run OpenAI analysis first!")
else:
    # 1. UNPACK JSON
    if isinstance(df['analysis'].iloc[0], dict):
        analysis_df = pd.json_normalize(df['analysis'])
        analysis_df['call_id'] = df['call_id'].values
        analysis_df['duration'] = df['duration'].values
        
        if 'language' in df.columns:
            analysis_df['language'] = df['language'].values
        else:
            analysis_df['language'] = 'unknown'
    else:
        analysis_df = df.copy()
        
        if 'language' not in analysis_df.columns:
            analysis_df['language'] = 'unknown'

    # Clean for viz
    analysis_df['duration'] = pd.to_numeric(analysis_df['duration'], errors='coerce')
    analysis_df['duration'] = analysis_df['duration'].fillna(analysis_df['duration'].mean())
    analysis_df = analysis_df.dropna(subset=['issue', 'sentiment'])

    print(f"📊 Viz data ready: {len(analysis_df)} calls analyzed")

    # CHART 1: ISSUE BAR CHART (Primary KPI)
    avg_issue = analysis_df.groupby(['issue', 'language'], as_index=False)['duration'].mean()

    fig1 = px.bar(
        avg_issue,
        x='issue',
        y='duration',
        color='language',
        barmode='group',
        title="Average Call Duration by Issue (100 Calls Analyzed)",
        labels={'duration': 'Avg Duration (seconds)', 'issue': 'Issue Type'}
    )
    fig1.update_layout(legend_title='Language', height=500)
    fig1.show()  # Interactive in Jupyter

    # CHART 2: SENTIMENT PIE (Bilingual Coverage)
    sentiment_counts = analysis_df['sentiment'].fillna('unknown').value_counts().reset_index()
    sentiment_counts.columns = ['sentiment', 'count']

    fig2 = px.pie(
        sentiment_counts,
        names='sentiment',
        values='count',
        title="Sentiment Distribution Across Calls",
        color_discrete_sequence=px.colors.qualitative.Set3
    )
    fig2.update_layout(height=500)
    fig2.show()

    # CHART 3: ACTION HEATMAP (Routing Matrix)
    heatmap_data = analysis_df.groupby(['issue', 'action']).size().unstack(fill_value=0)

    fig3 = px.imshow(
        heatmap_data.values,
        x=heatmap_data.columns,
        y=heatmap_data.index,
        title="Action Routing Heatmap (Issue → Action)",
        color_continuous_scale='RdYlGn',
        aspect="auto",
        labels=dict(color="Call Count")
    )
    fig3.update_layout(height=500)
    fig3.show()

    # CHART 4: DURATION TRENDS (Time Series)
    analysis_df = analysis_df.reset_index(drop=True)
    analysis_df['call_order'] = range(1, len(analysis_df) + 1)

    fig4 = px.scatter(
        analysis_df,
        x='call_order',
        y='duration',
        color='issue',
        size='duration',
        hover_name='summary',
        title="Call Duration Trends Over Time",
        labels={'call_order': 'Call Sequence', 'duration': 'Duration (s)'}
    )
    fig4.update_layout(height=500)
    fig4.show()

📊 Viz data ready: 100 calls analyzed
